# gRNAde Design for Eterna OpenKnot Benchmark

This notebook generates RNA sequence designs for the **Eterna OpenKnot Benchmark**, a community-wide, blinded competition for designing complex pseudoknotted RNA structures.

## Design Modes

This notebook supports two design scenarios:
1. **2D mode**: Secondary structure only (dot-bracket notation)
2. **3D mode**: Full 3D backbone coordinates from PDB files

## Pipeline Overview

1. **Load target structure**: PDB files and secondary structure (dot-bracket)
2. **Generate sequences**: Use gRNAde to sample ~1 million candidate designs
3. **Screen candidates**: RibonanzaNet predicts structure and SHAPE reactivity
4. **Filter designs**: Select top designs with OpenKnot Score ≥ 80
5. **Remove duplicates**: Ensure unique sequences for experimental synthesis

## Output

CSV file with designed sequences and computational scores:
- `sequence`: RNA sequence
- `openknot_score`: Predicted OpenKnot Score from RibonanzaNet
- `sc_score_ribonanzanet_ss`: Secondary structure self-consistency (MCC)
- `perplexity`: Model confidence (lower = more confident)
- Design metadata (temperature, seed, model checkpoint)

In [1]:
# Import libraries and set up the environment

import sys
sys.path.append('../../')

import dotenv
dotenv.load_dotenv("../../.env")

import os
import time
import random
from datetime import datetime
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F
from torchmetrics.functional.classification import binary_matthews_corrcoef

import lovely_tensors as lt
lt.monkey_patch()

from src.data.featurizer import RNAGraphFeaturizer
from src.data.sec_struct_utils import dotbracket_to_adjacency
from src.models import gRNAde
from src.evaluator import (
    openknot_score_ribonanzanet,
    self_consistency_score_ribonanzanet_sec_struct
)
from src.constants import NUM_TO_LETTER, PROJECT_PATH, DATA_PATH, FILL_VALUE

from main import set_seed

from tools.ribonanzanet.network import RibonanzaNet
from tools.ribonanzanet_sec_struct.network import RibonanzaNetSS

/home/ckj24/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/ckj24/.local/lib/python3.10/site-packages/Bio/Application/__init__.py:40: BiopythonDeprecationWarning: The Bio.Application modules and modules relying on it have been deprecated.

Due to the on going maintenance burden of keeping command line application
wrappers up to date, we have decided to deprecate and eventually remove these
modules.

We instead now recommend building your command line and invoking it directly
with the subprocess module.
  warnings.warn(


# Configure Design Scenario and Load gRNAde, RibonanzaNet Checkpoints

Configure the design task and model parameters:

- **PUZZLE_IDX**: Select puzzle from OpenKnot benchmark (0-19)
- **PUZZLE_SET**: Round 7a or 7b competition set. 7a -> Round 3, 7b -> Round 4 in the paper.
- **MODE**: `'2d'` (secondary structure only) or `'3d'` (with 3D coordinates)


In [ ]:
##################
# Design scenario
##################

# OpenKnot settings 
PUZZLE_IDX = 1  # 0-19
PUZZLE_SET = 'a' # 'a' or 'b'
MODE = '2d'    # '2d' or '3d'

# Single state or multi state design?
# - 1: single state
# - 2, 3, 5: multi state (number of states; currently unused)
max_num_conformers = 1

# random seed for reproducibility
seed = 42

# Default gRNAde model hyperparameters (configs/default.yaml; do not change unless you know what you're doing)
VERSION = 1.0
RADIUS = 0.0
TOP_K = 32
NUM_RBF = 32
NUM_POSENC = 32
NOISE_SCALE = 0.1
DROP_PROB_3D = 0.5
NODE_IN_DIM = (15, 4)
NODE_H_DIM = (128, 16)
EDGE_IN_DIM = (132, 3)
EDGE_H_DIM = (64, 4)
NUM_LAYERS = 4
DROP_RATE = 0.5
OUT_DIM = 4
DEFAULT_N_SAMPLES = 16
DEFAULT_TEMPERATURE = 0.1

In [ ]:
#########################################
# Initialise gRNAde model and featurizer
#########################################

# Set random seed
set_seed(seed)

# Set device (GPU/CPU)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Define data featurizer
print(f"Creating RNA graph featurizer for {MODE} design")
featurizer = RNAGraphFeaturizer(
    split = "test" if MODE == '3d' else "test_2d",
    radius = RADIUS,
    top_k = TOP_K,
    num_rbf = NUM_RBF,
    num_posenc = NUM_POSENC,
    max_num_conformers = max_num_conformers,
    noise_scale = NOISE_SCALE,
    drop_prob_3d = DROP_PROB_3D
)
# nucleotide mapping: {'A': 0, 'G': 1, 'C': 2, 'U': 3}

# Initialise model
print(f"Initialising gRNAde model")
model = gRNAde(
    node_in_dim = NODE_IN_DIM,
    node_h_dim = NODE_H_DIM, 
    edge_in_dim = EDGE_IN_DIM,
    edge_h_dim = EDGE_H_DIM, 
    num_layers = NUM_LAYERS,
    drop_rate = DROP_RATE,
    out_dim = OUT_DIM
)
# Load model checkpoint
model_path = os.path.join(PROJECT_PATH, "checkpoints", "gRNAde_drop3d@0.75_maxlen@500.h5")
print(f"Loading gRNAde2 checkpoint: {model_path}")
model.load_state_dict(torch.load(model_path, map_location=torch.device('cpu')))
# Transfer model to device in eval mode
model = model.to(device)
model.eval()
print()

#########################################
# Initialise other models for evaluation
#########################################

# Initialise RibonanzaNet for 1D self-consistency score
ribonanza_net = RibonanzaNet(
    os.path.join(PROJECT_PATH, 'tools/ribonanzanet/config.yaml'),
    os.path.join(PROJECT_PATH, 'checkpoints/ribonanzanet/ribonanzanet.pt'),
    device
)
# Transfer model to device in eval mode
ribonanza_net = ribonanza_net.to(device)
ribonanza_net.eval()
print()

# Initialise RibonanzaNetSS for 2D self-consistency score
ribonanza_net_ss = RibonanzaNetSS(
    os.path.join(PROJECT_PATH, 'tools/ribonanzanet_sec_struct/config.yaml'),
    os.path.join(PROJECT_PATH, 'checkpoints/ribonanzanet_sec_struct/ribonanzanet_ss.pt'),
    device
)
# Transfer model to device in eval mode
ribonanza_net_ss = ribonanza_net_ss.to(device)
ribonanza_net_ss.eval()
print()


Using device: cuda:0
Creating RNA graph featurizer for 2d design
Initialising gRNAde model
Loading gRNAde2 checkpoint: /home/ckj24/gRNAde_2/checkpoints/gRNAde_drop3d@0.75_maxlen@500.h5

Loading RibonanzaNet checkpoint: /home/ckj24/gRNAde_2/tools/ribonanzanet/ribonanzanet.pt

Loading RibonanzaNet SS checkpoint: /home/ckj24/gRNAde_2/tools/ribonanzanet_sec_struct/ribonanzanet_ss.pt



# Load Target Puzzle Metadata

Read puzzle specifications from the OpenKnot benchmark metadata file:

- **Puzzle ID**: Unique identifier for the RNA design target
- **Native sequence**: Wild-type sequence (if natural RNA) or reference sequence (if synthetic)
- **Target secondary structure**: Dot-bracket notation with pseudoknots

In [4]:
# Load metadata and secondary structures
df = pd.read_csv(f"metadata_7{PUZZLE_SET}.csv")
puzzle_id = list(df["puzzleID"].values)[PUZZLE_IDX]
native_seq = list(df["Sequence"].values)[PUZZLE_IDX]
target_sec_struct = list(df["Dot-bracket"].values)[PUZZLE_IDX]
print(f"puzzle_id: {puzzle_id}")
print(f"native_seq: {native_seq}")
print(f"target_sec_struct: {target_sec_struct}")

puzzle_id: 13620852
native_seq: UAUCAGUUAUAUGACUGACGGAACGUGGAAUUAACCACAUGAAGUAUAACGAUGACAAUGCCGACCGUCUGGGCG
target_sec_struct: .....(((((((..(.[[[[[...((((......))))..)..))))))).........(((..]]]]]..))).


In [8]:
# Create raw data object from PDB files
# (3D coordinates are randomly selected from the available conformers during design)
print("Target:", target_sec_struct)
pdb_filelist = os.listdir(f"structures/{puzzle_id}")
pdb_filelist = [f"structures/{puzzle_id}/{f}" for f in pdb_filelist]
print(f"Number of PDB files found: {len(pdb_filelist)}")
for pdb_file, sec_struct in zip(pdb_filelist, raw_data["sec_struct_list"]):
    print(f'\n{pdb_file.split("/")[-1].split(".")[0]}: {sec_struct}')
    print("Correlation: ",
        float(binary_matthews_corrcoef(
            torch.tensor(
                dotbracket_to_adjacency(sec_struct, keep_pseudoknots=True)
            ),
            torch.tensor(
                dotbracket_to_adjacency(target_sec_struct, keep_pseudoknots=True)
            )
        ))
    )
_, raw_data = featurizer.featurize_from_pdb_filelist(pdb_filelist)
raw_data

Target: .....(((((((..(.[[[[[...((((......))))..)..))))))).........(((..]]]]]..))).
Number of PDB files found: 2

trRosetta: .....(((((.((.((..[[[.....((.....{))...)).)).)))).)........(((..]]].}..))).
Correlation:  0.7360913157463074

RFdiff_0: .....((((((((.(.[[[[[...((((......))))..).)))))))).........(((..]]]]]..))).
Correlation:  0.9757252931594849


{'sequence': 'UAUCAGUUAUAUGACUGACGGAACGUGGAAUUAACCACAUGAAGUAUAACGAUGACAAUGCCGACCGUCUGGGCG',
 'coords_list': [tensor[75, 3, 3] n=675 (2.6Kb) x∈[-28.544, 35.562] μ=-0.138 σ=12.606,
  tensor[75, 3, 3] n=675 (2.6Kb) x∈[-30.498, 26.704] μ=0.027 σ=11.543],
 'sec_struct_list': ['.....(((((.((.((..[[[.....((.....{))...)).)).)))).)........(((..]]].}..))).',
  '.....((((((((.(.[[[[[...((((......))))..).)))))))).........(((..]]]]]..))).']}

# Generate Design Library with Computational Filtering

## Generation Strategy

This cell implements the complete gRNAde pipeline from Figure 1 of the paper:

### A. Generation Stage (per batch)
- **Total samples**: 1,000,000 candidate sequences
- **Batch size**: 128 designs per iteration
- **Temperature**: Random sampling from [0.1, 1.0] for sequence diversity
- **Output**: Sequence samples + model confidence (perplexity)

### B. Screening Stage  
For each generated sequence:
1. **OpenKnot Score** (RibonanzaNet):
   - Predicts SHAPE chemical reactivity profile and compares to target secondary structure
   - Used as initial filter to reduce I/O (threshold ≥ 80 for efficiency)

2. **Secondary Structure Self-Consistency** (RibonanzaNetSS):
   - Predicts pseudoknotted secondary structure from sequence
   - Computes MCC between predicted and target structure

### C. Selection Stage
- Designs are saved to CSV and duplicates are removed at the end of generation
- Each design includes full metadata: sequence, edit distance, perplexity, self-consistency scores
- Target: Generate ~100,000 total designs that pass the initial OpenKnot score threshold of 80

## Output Format
CSV columns: `fasta_desc, sequence, model, seed, temperature, edit_dist, perplexity, sc_score_ribonanzanet, sc_score_ribonanzanet_ss`

In [ ]:
# Total number of samples to generate
total_samples = 1_000_000

# Number of designed samples (per batch)
n_samples = 128

# Number of designs which need to pass filter
n_pass = 100_000
pass_threshold = 80  # openknot score threshold

# Create output directory
current_datetime = datetime.now().strftime("%Y%m%d_%H%M%S")
output_dir = f"designs/"
os.makedirs(output_dir, exist_ok=True)
output_file = f"{output_dir}/{puzzle_id}_{MODE}_{current_datetime}.csv"
with open(output_file, 'w') as f_out:
    f_out.write("fasta_desc,sequence,model,id,seed,temperature,perplexity,openknot_score,sc_score_ribonanzanet_ss\n")

# create raw data for secondary structure-only mode
if MODE == '2d':
    raw_data_2d = {
        "sequence": native_seq,
        "coords_list": [torch.ones(len(target_sec_struct), 3, 3) * FILL_VALUE],
        "sec_struct_list": [target_sec_struct],
    }
    # featurize only once
    featurized_data = featurizer(raw_data_2d).to(device)

# collate designed sequences for saving to csv
designs = []
t = tqdm(range(total_samples // n_samples))
for _ in t:
    t.set_description(f"Total designs: {len(designs)}")

    # Set temperature and seed
    temperature = np.random.uniform(0.1, 1.0)
    seed = random.randint(0, 9999)
    set_seed(seed)

    if MODE == '3d':
        # Featurize raw data, select random conformer from available 3D structures
        featurized_data = featurizer(raw_data).to(device)
    else:
        pass  # already featurized above one-time for 2D mode
    
    # sample n_samples from model for single data point: n_samples x seq_len
    samples, logits = model.sample(featurized_data, n_samples, temperature, return_logits=True)

    # perplexity per sample: n_samples x 1
    n_nodes = logits.shape[1]
    perplexity = (
        torch.exp(
            F.cross_entropy(
                logits.view(n_samples * n_nodes, model.out_dim),
                samples.view(n_samples * n_nodes).long(),
                reduction="none",
            )
            .view(n_samples, n_nodes)
            .mean(dim=1)
        )
        .cpu()
        .numpy()
    )

    # openknot score per sample: n_samples x 1
    openknot_scores = openknot_score_ribonanzanet(
        samples.cpu().numpy(), 
        target_sec_struct, 
        featurized_data.mask_seq.cpu().numpy(),
        ribonanza_net
    )

    # 2D self-consistency score per sample: n_samples x 1
    sc_score_ribonanzanet_ss = self_consistency_score_ribonanzanet_sec_struct(
        samples.cpu().numpy(),
        target_sec_struct,
        featurized_data.mask_seq.cpu().numpy(),
        ribonanza_net_ss,
    )

    # only keep designs which pass openknot score threshold to reduce I/O
    idx_good_designs = (openknot_scores >= pass_threshold)
    if np.sum(idx_good_designs) > 0:
        samples = samples.cpu().numpy()[idx_good_designs]
        perplexity = perplexity[idx_good_designs]
        openknot_scores = openknot_scores[idx_good_designs]
        sc_score_ribonanzanet_ss = sc_score_ribonanzanet_ss[idx_good_designs]
        
        # collate designed sequences in fasta and csv format
        with open(output_file, 'a') as f_out:
            for zipped in zip(samples, perplexity, openknot_scores, sc_score_ribonanzanet_ss):
                seq, perp, ok_score, sc_score = zipped
                seq = "".join([NUM_TO_LETTER[num] for num in seq])
                design = [
                    f"{os.path.split(model_path)[-1]} puzzle_id={puzzle_id} seed={seed} temperature={temperature} perplexity={perp:.4f} openknot_score={ok_score:.4f} sc_score_ribonanzanet_ss={sc_score:.4f}",
                    seq, os.path.split(model_path)[-1], puzzle_id, seed, temperature, perp, ok_score, sc_score
                ]
                designs.append(design)
                f_out.write(",".join(map(str, design)) + "\n")

    # Stop if enough designs have been collected
    if len(designs) >= n_pass:
        break


In [ ]:
# Load all designs and remove duplicates
all_designs_df = pd.read_csv(output_file)
print("Total designs:", len(all_designs_df))
all_designs_df = all_designs_df.drop_duplicates(subset="sequence")
print("Unique designs:", len(all_designs_df))
# overwrite csv file with unique designs only
all_designs_df.to_csv(output_file, index=False)